# ML-10 - Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/8-BitRhyon/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

Lane 4: CTR opportunity scoring for content review.
This notebook turns the validated output into a content action playbook: ranked actions with reason
codes, an archetype to action map, the decay/refresh insight, intended use, limits, human review,
cost and value, monitoring triggers, and a clear no-go list.

The validated output is the **value rule**, not the model. ML-08 showed the model does not beat the
rule at the 140-slot budget, and ML-09 showed the model's precision@140 falls to 0.886 under a
time-forward split while the rule holds at 0.979. The playbook therefore uses the rule.

Data: March 2026 page-months from the warehouse, scored on April 1. June 2026 stays sealed.

This is a decision-support playbook, not a production system. No automation, no SLA.


## 1. Ranked actions + reason codes

**The rule.** A page is worth reviewing when it gets enough impressions to matter and its CTR is
below the median for its position tier. The score is the click gap against that median:

```
score = max(0, tier_median_ctr - page_ctr) * impressions / 100
```

**In scope.** `top_3`, `page_1`, and `page_2` pages with at least 500 March impressions and 20 days
of search data. Deep pages are out of scope: their constraint is ranking, not the snippet, so a CTR
review spends a slot without addressing the cause.

**Archetypes to actions.**

| Archetype | Condition | Reason code | Action | Why this action |
|---|---|---|---|---|
| Near-zero CTR at volume | `imp >= 100,000` and `ctr < 0.02%` | `VERIFY_FIRST` | Check tracking and query mix before any edit | A 0.001% CTR on 134k impressions is more often a measurement or zero-click query problem than a bad title |
| Aged snippet gap | `age >= 270 days` | `CTR_GAP_AGED` | Refresh the page, then rewrite the snippet | The research paper's decay curve puts 271-365 days at the low point. A snippet-only edit leaves stale content in place. |
| Prime snippet gap | `top_3` or `page_1`, `age < 270` | `CTR_GAP_PRIME` | Rewrite the title and meta description | The page already ranks, so the snippet is the remaining lever |

The decay/refresh insight is the middle row. Age is not a second score. It decides which action a
snippet gap receives. Pages in the decay zone get a refresh plus a snippet, not a snippet alone.

**The queue.** The top 140 slots cover the 7-week review capacity from ML-02, at 20 reviews a week.


In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

import duckdb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Resolve the repo root so exports land in <repo>/work/outputs and <repo>/work/figures.
REPO_URL = "https://github.com/8-BitRhyon/flyrank-ml-internship"
if "google.colab" in sys.modules and not Path("work/notebooks").is_dir():
    if not Path("flyrank-ml-internship").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
while not Path("work/notebooks").is_dir() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    try:
        from huggingface_hub import get_token
        HF_TOKEN = get_token()
    except Exception:
        HF_TOKEN = None

assert HF_TOKEN, "No Hugging Face token. Set a Colab Secret named HF_TOKEN, or run `hf auth login` once."

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

BASE = "hf://datasets/FlyRank/internship-warehouse"
K = 140
FLOOR = 500
SEALED_MONTH = "2026-06"

month = con.execute(f"""
    WITH d AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp,
               SUM(gsc_clicks) AS clk,
               SUM(gsc_sum_position) AS pos_sum,
               SUM(CASE WHEN gsc_avg_position > 0 THEN gsc_impressions END) AS pos_imp,
               COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS days_gsc
        FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        GROUP BY 1, 2
    )
    SELECT client_hash_id, content_hash_id, imp, clk, days_gsc,
           pos_sum / NULLIF(pos_imp, 0) AS avg_pos,
           100.0 * clk / NULLIF(imp, 0) AS ctr_pct
    FROM d
    ORDER BY client_hash_id, content_hash_id
""").df()

dim = con.execute(
    f"SELECT client_hash_id, content_hash_id, content_created_date "
    f"FROM read_parquet('{BASE}/dim_content.parquet')"
).df()

frame = month[(month.imp >= FLOOR) & (month.days_gsc >= 20) & month.avg_pos.notna()].copy()
frame["tier"] = pd.cut(
    frame.avg_pos, [0, 3, 10, 20, np.inf], labels=["top_3", "page_1", "page_2", "deep"]
).astype(str)
tier_median = frame.groupby("tier").ctr_pct.median()

frame = frame.merge(dim, on=["client_hash_id", "content_hash_id"], how="left")
frame["age_days"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(frame.content_created_date)).dt.days
frame["gap"] = ((frame.tier.map(tier_median) - frame.ctr_pct).clip(lower=0) * frame.imp / 100).round(2)


def assign_archetype(row):
    if row.imp >= 100_000 and row.ctr_pct < 0.02:
        return "VERIFY_FIRST"
    if row.age_days >= 270:
        return "CTR_GAP_AGED"
    return "CTR_GAP_PRIME"


ACTION = {
    "VERIFY_FIRST": "Check tracking and query mix before any edit",
    "CTR_GAP_AGED": "Refresh the page, then rewrite the snippet",
    "CTR_GAP_PRIME": "Rewrite the title and meta description",
}

in_scope = frame[frame.tier != "deep"].copy()
in_scope["archetype"] = in_scope.apply(assign_archetype, axis=1)
in_scope["reason_code"] = in_scope.archetype
in_scope["action"] = in_scope.archetype.map(ACTION)

queue = in_scope.sort_values(["gap", "imp"], ascending=[False, False]).reset_index(drop=True)
queue["rank"] = queue.index + 1

print(f"March page-months: {len(frame):,} | in scope for CTR work: {len(queue):,}")
print(f"tier medians: {tier_median.round(4).to_dict()}")
print(f"total gap in scope: {queue.gap.sum():,.0f} clicks")
print(f"top {K}: {queue.head(K).gap.sum():,.0f} clicks "
      f"({100 * queue.head(K).gap.sum() / queue.gap.sum():.1f}% of the in-scope gap)")
print("\ntop 140 by archetype:")
print(queue.head(K).archetype.value_counts().to_string())


March page-months: 59,306 | in scope for CTR work: 48,725
tier medians: {'deep': 0.0741, 'page_1': 0.2109, 'page_2': 0.1642, 'top_3': 0.2322}
total gap in scope: 96,691 clicks
top 140: 11,716 clicks (12.1% of the in-scope gap)

top 140 by archetype:
archetype
CTR_GAP_PRIME    100
CTR_GAP_AGED      36
VERIFY_FIRST       4


**Reading the queue.** The 140 slots take 100 prime snippet gaps, 36 aged gaps, and 4
`VERIFY_FIRST` pages. The mix matters more than the count: 36 of 140 reviews are refreshes, not
snippet edits, which is the decay insight turning into a workload estimate. Two more pages are older
than 270 days but are classified `VERIFY_FIRST` first, because their near-zero CTR should be checked
before any edit.

The score is an upper bound. It says what the page would gain if it reached its tier median, not
what a rewrite will deliver. Use it to order the queue, not to promise a result.


## 2. Intended use and limits

**Intended use.** A content editor with 140 review slots over 7 weeks uses the queue to decide
which pages to open first. The queue is decision-support: it orders pages by the size of the CTR gap
against their peers, weighted by traffic. It is not a forecast of recovered clicks and it is not a
client-facing deliverable.

**Limits, measured.**

- **Coverage.** 48,725 of 331,437 March page-months (14.7%) are in scope. The rest are below the impression floor, have no position data, or are deep pages where ranking is the constraint.
- **One month, one definition.** The queue is March 2026. Tier medians move month to month, so the queue must be recomputed, not reused.
- **Tier-relative label.** A page with a healthy absolute CTR can be flagged if it falls below a low tier median. The deep tier median is about 0.07%. A page at 0.5% that drops to 0.06% is still a large relative fall.
- **No engagement.** GA4 is available on 4.2% of March rows, so engagement columns are excluded. The playbook says nothing about on-page experience.
- **No causal claim.** The queue identifies an association between position, CTR, and headroom. It does not prove that a rewrite recovers the gap.
- **Honest performance.** ML-09 measured the rule at precision@140 0.979 on a same-period split. That is the number this playbook inherits, not a guarantee for April.
- **No revenue.** The score is in clicks. Attaching a value per click is the team's decision, not the model's.


In [2]:
total_page_months = int(con.execute(
    f"""
    WITH pm AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp,
               COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS days_gsc
        FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        GROUP BY 1, 2
    )
    SELECT COUNT(*) FROM pm
    """
).fetchone()[0])

ga4_share = con.execute(f"""
    SELECT 100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*)
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").fetchone()[0]

coverage = pd.DataFrame([
    {"measure": "March page-months", "value": total_page_months},
    {"measure": "in scope for CTR work", "value": len(queue)},
    {"measure": "coverage of March page-months, %", "value": round(100 * len(queue) / total_page_months, 1)},
    {"measure": "GA4 available, % of March rows", "value": round(ga4_share, 2)},
    {"measure": "deep pages excluded from CTR work", "value": int((frame.tier == "deep").sum())},
    {"measure": "sealed month used", "value": SEALED_MONTH},
])
coverage


,measure,value
0,March page-months,331437
1,in scope for CTR work,48725
2,"coverage of March page-months, %",14.7
3,"GA4 available, % of March rows",4.21
4,deep pages excluded from CTR work,10581
5,sealed month used,2026-06


## 3. Human review + the no-go list

**What a person checks before acting.**

1. **Query report.** Are the impressions from queries this page can answer? A zero-click query mix
   (definitions, direct answers, navigational) will not respond to a better title.
2. **SERP.** Is a featured snippet, AI answer, or shopping block taking the click? If so, the gap is
   structural, not editorial.
3. **Tracking.** For `VERIFY_FIRST` pages, confirm the clicks are really zero before spending a slot.
4. **Recency.** Has the page been refreshed or rewritten in the last 30 days? If yes, wait for the
   effect before touching it again.
5. **Ownership.** Does the client have a scheduled change or a known migration? Do not compete with it.

**The no-go list. What should never be automated.**

- No automatic title or meta changes, and no auto-publishing.
- No bulk edits without a sample review of at least 10% of the batch.
- No client-facing promises, revenue estimates, or ranking guarantees.
- No use of the queue in editor performance reviews. The score measures pages, not people.
- No pages below the 500-impression floor, and no deep pages in a CTR queue.
- No June 2026 data in development. It stays sealed.
- No use of the queue as a production ranking system. It is a monthly planning artifact.

The rule is intentionally simple so an editor can read the reason code and disagree with it. That
is the point of a baseline.


In [3]:
review = queue.head(K).copy()
checks = pd.DataFrame([
    {"check": "pages in top 140", "value": len(review)},
    {"check": "VERIFY_FIRST before editing", "value": int((review.archetype == "VERIFY_FIRST").sum())},
    {"check": "aged pages needing refresh", "value": int((review.archetype == "CTR_GAP_AGED").sum())},
    {"check": "prime pages needing snippet only", "value": int((review.archetype == "CTR_GAP_PRIME").sum())},
    {"check": "top-3 pages", "value": int((review.tier == "top_3").sum())},
    {"check": "page-1 pages", "value": int((review.tier == "page_1").sum())},
    {"check": "page-2 pages", "value": int((review.tier == "page_2").sum())},
    {"check": "pages older than 270 days", "value": int((review.age_days >= 270).sum())},
    {"check": "pages with zero clicks", "value": int((review.clk == 0).sum())},
])
checks


,check,value
0,pages in top 140,140
1,VERIFY_FIRST before editing,4
2,aged pages needing refresh,36
3,prime pages needing snippet only,100
4,top-3 pages,33
5,page-1 pages,102
6,page-2 pages,5
7,pages older than 270 days,38
8,pages with zero clicks,4


## 4. Monitoring / retrain triggers

The playbook has no model in production, so monitoring is about the rule and the data, not about
drift in weights.

| Trigger | Threshold | What to do |
|---|---|---|
| Monthly recompute | every month | Rebuild tier medians and the queue. Never reuse last month's ranks. |
| Queue overlap with the previous month | below 50% of the top 140 | Check for a tracking, indexing, or position-pipeline change before trusting the new queue. |
| Near-zero share in the top 140 | above 10% | Run a tracking audit before assigning reviews. |
| Deep-page share in the in-scope queue | above 10% | Check the position pipeline. The tier cut may be misreading positions. |
| Next-month precision@140 | below 0.90 | Revisit the rule. The same-period range was 0.95 to 0.98 in ML-08, so a drop below 0.90 is a real signal. |
| GA4 coverage | above 50% of rows | Revisit the engagement exclusion. The lane could add an engagement arm. |
| Any model adoption | quarterly | Retrain and rerun the time-forward audit from ML-09 before shipping. |

These are planning triggers, not production alerts. A small team can run them in a monthly review.


In [4]:
cum = queue.gap.cumsum() / queue.gap.sum()
top140_gap = queue.head(K).gap.sum()
top500_gap = queue.head(500).gap.sum()

monitors = pd.DataFrame([
    {"monitor": "current top-140 gap", "value": round(top140_gap, 1), "unit": "clicks per month"},
    {"monitor": "current top-500 gap", "value": round(top500_gap, 1), "unit": "clicks per month"},
    {"monitor": "near-zero share in top 140, %",
     "value": round(100 * (queue.head(K).archetype == "VERIFY_FIRST").mean(), 2), "unit": "%"},
    {"monitor": "aged share in top 140, %",
     "value": round(100 * (queue.head(K).archetype == "CTR_GAP_AGED").mean(), 2), "unit": "%"},
    {"monitor": "median age in top 140", "value": int(queue.head(K).age_days.median()), "unit": "days"},
])
monitors


,monitor,value,unit
0,current top-140 gap,11716.10,clicks per month
1,current top-500 gap,23497.60,clicks per month
2,"near-zero share in top 140, %",2.86,%
3,"aged share in top 140, %",25.71,%
4,median age in top 140,155.00,days


## 5. Exports for the paper

Three exports, with different git rules:

- `work/outputs/action_playbook_queue.csv` is the ranked queue. It stays out of git by design: the CI leak guard blocks data files, and the notebook regenerates it.
- `work/figures/ml10_gap_capture.png` and `work/figures/ml10_queue_composition.png` are reused in the paper, so they are committed.
- `work/outputs/ml10_playbook_metrics.json` is the receipt for the numbers quoted in the paper, so it is committed.

**Cost and value.** At 1.5 hours per review, 140 reviews is about 210 editor-hours. The top 140
carries about 11,700 clicks of expected gap, roughly 84 clicks per review. Extending to 500 reviews
adds about 11,800 clicks for 360 more reviews, roughly 33 clicks per review. The first 140 are the
better use of a small team's time, which is why the capacity assumption from ML-02 is the default.


In [5]:
out_dir = Path("work/outputs")
fig_dir = Path("work/figures")
out_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

EXPORT_COLS = ["rank", "content_hash_id", "client_hash_id", "tier", "imp", "clk", "ctr_pct",
               "avg_pos", "age_days", "gap", "archetype", "reason_code", "action"]
queue[EXPORT_COLS].to_csv(out_dir / "action_playbook_queue.csv", index=False)

# Figure 1: cumulative share of the gap.
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(np.arange(1, len(cum) + 1), 100 * cum, color="#1f4e79", lw=2)
ax.set_xscale("log")
ax.axvline(K, color="#c00000", ls="--", lw=1)
ax.axvline(500, color="#7f7f7f", ls=":", lw=1)
ax.text(K, 5, f"  {K} reviews", color="#c00000", fontsize=9)
ax.text(500, 5, "  500 reviews", color="#7f7f7f", fontsize=9)
ax.set_xlabel("pages reviewed (log scale)")
ax.set_ylabel("share of in-scope click gap captured, %")
ax.set_title("The first 140 pages carry 12% of the gap; the curve flattens quickly")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(fig_dir / "ml10_gap_capture.png", dpi=150)
plt.close(fig)

# Figure 2: top-140 composition by archetype and tier.
review = queue.head(K)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
review.archetype.value_counts().plot.barh(ax=axes[0], color="#1f4e79")
axes[0].set_title("Top 140 by action archetype")
axes[0].set_xlabel("pages")
review.tier.value_counts().plot.barh(ax=axes[1], color="#2e7d32")
axes[1].set_title("Top 140 by position tier")
axes[1].set_xlabel("pages")
for ax in axes:
    ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(fig_dir / "ml10_queue_composition.png", dpi=150)
plt.close(fig)

metrics = {
    "month": "2026-03",
    "sealed_month": SEALED_MONTH,
    "capacity_reviews": K,
    "in_scope_pages": int(len(queue)),
    "march_page_months": total_page_months,
    "coverage_pct": round(100 * len(queue) / total_page_months, 2),
    "ga4_share_pct": round(float(ga4_share), 2),
    "total_gap_clicks": round(float(queue.gap.sum()), 2),
    "top140_gap_clicks": round(float(top140_gap), 2),
    "top140_share_pct": round(100 * float(top140_gap) / float(queue.gap.sum()), 2),
    "top500_gap_clicks": round(float(top500_gap), 2),
    "top140_archetypes": {str(k): int(v) for k, v in review.archetype.value_counts().items()},
    "top140_tiers": {str(k): int(v) for k, v in review.tier.value_counts().items()},
    "top140_median_age_days": int(review.age_days.median()),
    "editor_hours_at_1_5h_per_review": int(K * 1.5),
    "clicks_per_review_top140": round(float(top140_gap) / K, 1),
    "clicks_per_review_141_to_500": round(float(top500_gap - top140_gap) / (500 - K), 1),
}
(out_dir / "ml10_playbook_metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")

print(f"wrote {out_dir / 'action_playbook_queue.csv'} ({len(queue):,} rows)")
print(f"wrote {fig_dir / 'ml10_gap_capture.png'}")
print(f"wrote {fig_dir / 'ml10_queue_composition.png'}")
print(f"wrote {out_dir / 'ml10_playbook_metrics.json'}")
print(json.dumps(metrics, indent=2))


wrote work/outputs/action_playbook_queue.csv (48,725 rows)
wrote work/figures/ml10_gap_capture.png
wrote work/figures/ml10_queue_composition.png
wrote work/outputs/ml10_playbook_metrics.json
{
  "month": "2026-03",
  "sealed_month": "2026-06",
  "capacity_reviews": 140,
  "in_scope_pages": 48725,
  "march_page_months": 331437,
  "coverage_pct": 14.7,
  "ga4_share_pct": 4.21,
  "total_gap_clicks": 96691.17,
  "top140_gap_clicks": 11716.08,
  "top140_share_pct": 12.12,
  "top500_gap_clicks": 23497.55,
  "top140_archetypes": {
    "CTR_GAP_PRIME": 100,
    "CTR_GAP_AGED": 36,
    "VERIFY_FIRST": 4
  },
  "top140_tiers": {
    "page_1": 102,
    "top_3": 33,
    "page_2": 5
  },
  "top140_median_age_days": 155,
  "editor_hours_at_1_5h_per_review": 210,
  "clicks_per_review_top140": 83.7,
  "clicks_per_review_141_to_500": 32.7
}


## Self-check

- [x] Ranked actions with reason codes, built from the validated rule and not the weaker model
- [x] Archetype to action mapping, including the decay and refresh insight
- [x] Intended use stated, and limits measured rather than asserted
- [x] Human review steps and a no-go list that says what must never be automated
- [x] Cost and value per review at 140 and at 500 reviews
- [x] Monitoring and retrain triggers with thresholds
- [x] Queue exported to `work/outputs/`, figures to `work/figures/`, metrics JSON committed
- [x] Public-safe language: observed, measured, directional, decision-support
- [x] The notebook ran top to bottom with no errors (executed before commit)
- [x] No client names, URLs, private queries, or tokens anywhere
- [x] Committed under `work/notebooks/`
